# 11（進階）時序基礎模型實作：TimesFM 3.0 零樣本預測

Ch11 的 [`11_dl_sequence.ipynb`](11_dl_sequence.ipynb) 我們**親手訓練**了 LSTM / CNN，才打敗 persistence 基準。這本 notebook 換一個問題：

> 如果有一個模型**已經在數千億個時間點上預訓練好**，我們一行訓練程式都不寫，直接拿來預測，會比較好嗎？

這就是**時序基礎模型（Time Series Foundation Model, TSFM）**的**零樣本（zero-shot）預測**。我們用 Google 在 2026 年 8 月發布的 **TimesFM 3.0** 實際跑一次，而且**誠實評估**：

**流程：零樣本是什麼 → 載入模型 → 預測區間 → 跟 Ch11 同一個測試期做滾動起點回測 → 加入氣溫共變量 → 換一條 Ch07 的監測序列 → 什麼時候不能相信它 → 授權（重要！）**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# Colab 額外步驟：安裝 TimesFM 3.0（本機用 uv 的話：uv sync --group timesfm）
import sys
if 'google.colab' in sys.modules:
    !pip install -q "timesfm[torch]>=3.0.2"

## 零樣本（zero-shot）是什麼意思？

Ch11 Part A 的 LSTM 要先 `fit`：拿這條序列的前 300 天反覆練習幾百個 epoch，才學會「7 天前的氣溫會推高病例數」。

TSFM 反過來：Google 先用大量、跨領域的時間序列（維基百科瀏覽量、Google Trends、公開預測資料集、合成資料……）把模型訓練好，**你拿到的是已經訓練完的權重**。預測時只要把「歷史」丟進去，模型直接吐出未來——**沒有 `fit()` 這一步**：

| | 自己訓練（Ch11 LSTM） | 零樣本 TSFM（TimesFM 3.0） |
|---|---|---|
| 要不要訓練 | 要，每條序列都要 | 不用 |
| 需要多長歷史 | 數百個時間窗口才穩 | 幾十個點就能預測（但越多通常越好） |
| 輸出 | 一個點預測 | 中位數 + 9 個分位數（0.1–0.9）→ **預測區間** |
| 懂這條序列的特殊機制嗎 | 能學到（如 7 天延遲） | 只能從歷史形狀「猜」 |
| 硬體 | 筆電秒級 | 下載約 1.3 GB 權重、約 3 GB 記憶體 |

> 🧭 **本 notebook 的兩條執行路徑**：在 Colab 或有安裝 `timesfm` 的電腦上，下面每一格都**真的呼叫 TimesFM 3.0**。本書網站在建置（build）時不下載 1.3 GB 的模型，而是由 `epi_learning.tsfm` 讀取一次真實執行後存下的預測結果（`data/tsfm_cache/ch11_timesfm.json`）——每一格都會印出它走的是哪條路徑，數字跟真實執行相同（只四捨五入到小數 3 位）。

In [ ]:
# --- Step 1：載入套件、重建 Ch11 的「登革熱 × 氣溫」合成序列 ---
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from epi_learning import tsfm

# -- CJK font setup（避免中文標籤顯示為方框）--
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- 跟 11_dl_sequence.ipynb 完全相同的資料產生流程（同一個 seed）--
np.random.seed(1)
n = 360
t = np.arange(n)
temp = 24 + 7 * np.sin(2 * np.pi * (t - 30) / 365) + np.random.normal(0, 1.0, n)
drive = np.clip(temp - 24, 0, None)
cases = np.zeros(n)
for i in range(n):
    lag = cases[i - 1] if i >= 1 else 0
    cases[i] = max(
        0,
        0.55 * lag
        + 3.2 * (drive[i - 7] if i >= 7 else 0)
        + 4 * np.sin(2 * np.pi * t[i] / 7)
        + 6
        + np.random.normal(0, 2.0),
    )

H = 7            # 跟 Ch11 一樣：預測 7 天後
split = n - 60   # 跟 Ch11 一樣：最後 60 天是測試期
print(f"資料長度：{n} 天，測試期：day {split}–{n - 1}")
print(f"病例數：mean={cases.mean():.1f}, min={cases.min():.1f}, max={cases.max():.1f}")

## Step 2 — 載入 TimesFM 3.0

官方 API 只有三行：`ModelConfig` 指定權重（Hugging Face 上的 `google/timesfm-3.0-pytorch`）、批次大小與裝置，`TimesFM3Evaluator` 載入模型。第一次執行會下載約 1.3 GB 權重到 `~/.cache/huggingface`，之後直接讀硬碟。

`TimesFM3Evaluator` 是官方評測用的預設組合：一定回傳分位數、做「對稱平均」（把序列正負翻轉各預測一次再平均，比較穩）、而且歷史全是非負值時，預測也不會出現負數——很適合病例數。

In [ ]:
# --- Step 2：載入 TimesFM 3.0（沒有安裝 timesfm 時改讀快取）---
CACHE = tsfm.cache_path("ch11_timesfm")   # data/tsfm_cache/ch11_timesfm.json
MODE = tsfm.resolve_mode()                # "model"：有裝 timesfm；"cache"：沒裝（例如本書網站的建置）

if MODE == "cache":
    model = None
    print("路徑 = 快取：這次沒有執行 timesfm3，以下預測讀自一次真實的 TimesFM 3.0 執行")
    print(tsfm.describe_cache(CACHE))
else:
    import time
    from timesfm3 import TimesFM3Evaluator, ModelConfig

    t0 = time.perf_counter()
    config = ModelConfig(
        checkpoint_path="google/timesfm-3.0-pytorch",  # Hugging Face 上的權重（約 1.3 GB）
        per_core_batch_size=32,                        # 一次前向傳播處理幾條序列
        device=tsfm.default_device(),                  # 有 GPU 用 "cuda"，否則 "cpu"
    )
    model = TimesFM3Evaluator(config)
    n_params = sum(p.numel() for p in model.model.parameters())
    print(f"路徑 = 模型：TimesFM 3.0 載入完成，{n_params / 1e6:.0f}M 個參數，"
          f"裝置 {model.device}，耗時 {time.perf_counter() - t0:.0f} 秒")

## Step 3 — 第一次零樣本預測：給它 300 天，請它預測接下來 60 天

`model.predict_batch()` 吃一個「序列清單」（每條可以不同長度），回傳每條序列的：

- `.forecast`：中位數預測，形狀 `(horizon,)`
- `.quantiles`：9 個分位數（0.1, 0.2, …, 0.9），形狀 `(horizon, 9)`——第 0 欄與第 8 欄之間就是 **80% 預測區間**

下面先直接呼叫一次 `model.predict_batch()`，再透過 `tsfm.forecast_batch()` 做同一件事（它在有模型時就是呼叫 `predict_batch`，沒有模型時讀快取）。

In [ ]:
# --- Step 3：零樣本預測——前 300 天 → 未來 60 天 ---
context = cases[:split]   # 模型看得到的歷史：day 0–299（沒有任何訓練）
H_LONG = 60

if model is not None:
    # 真正的 TimesFM 3.0 呼叫：沒有 fit()，直接預測
    out = next(model.predict_batch([context.astype(np.float32)], horizon=H_LONG, return_quantiles=True))
    print("forecast 形狀：", out.forecast.shape, "| quantiles 形狀：", out.quantiles.shape)

first = tsfm.forecast_batch(model, [context], H_LONG, cache_key="first_look", cache_file=CACHE)
median = first.forecast[0]
lo, hi = first.quantile(0.1)[0], first.quantile(0.9)[0]
days = np.arange(split, split + H_LONG)

print(f"分位數層級：{tsfm.QUANTILE_LEVELS}")
print(f"未來 60 天中位數預測的 MAE：{np.mean(np.abs(median - cases[days])):.2f}")
print(f"真實值落在 80% 預測區間（0.1–0.9 分位數）內的比例：{tsfm.coverage(cases[days], lo, hi):.0%}（理想約 80%）")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(np.arange(200, split), cases[200:split], color="#6B6B6B", label="模型看得到的歷史")
ax.plot(days, cases[days], color="#1A1A1A", linewidth=2, label="真實病例數（模型沒看過）")
ax.plot(days, median, color="#D97757", linewidth=2, label="TimesFM 中位數預測")
ax.fill_between(days, lo, hi, color="#D97757", alpha=0.2, label="80% 預測區間")
ax.axvline(split, color="#6A9BCC", linestyle="--")
ax.set_xlabel("天數（day index）")
ax.set_ylabel("每日病例數")
ax.set_title("TimesFM 3.0 零樣本預測：沒有訓練，直接預測 60 天")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Step 4 — 誠實評估：跟 Ch11 用同一個測試期、同一套指標

一張漂亮的圖不算數。要跟 Ch11 的 LSTM / CNN 公平比較，條件必須完全一樣：

- **同一條序列、同一個測試期**：day 300–359，共 60 個目標日
- **同一個任務**：用「到起點為止」的歷史，預測 **7 天後**的病例數
- **滾動起點回測（rolling-origin backtest）**：每個目標日都有自己的起點，模型只看得到起點之前的資料——這就是 Ch11 第 6 節「時間序列切分」的精神，未來絕不能漏進來
- **同一套指標**：MAE、RMSE、MAPE；TimesFM 額外有預測區間，所以再看**覆蓋率**和**加權分位數損失（WQL，CRPS 的近似）**

LSTM / CNN 的數字直接引用 [`11_dl_sequence.ipynb`](11_dl_sequence.ipynb) 在同一個測試期印出的結果；persistence 在這裡重算一次，用來確認測試期真的一模一樣（應該是 2.879）。

In [ ]:
# --- Step 4：滾動起點回測（60 個起點，每個預測 7 天）---
origins = np.arange(split - H + 1, n - H + 1)              # 起點 day 294–353
targets = origins + H - 1                                   # 第 7 天 = day 300–359（= Ch11 的 idxte）
contexts = tsfm.rolling_origin_contexts(cases, origins)     # [cases[:294], cases[:295], ...]

bt = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_univariate", cache_file=CACHE)
tfm_pred = bt.forecast[:, H - 1]                            # 每個起點的「第 7 天」中位數預測
tfm_lo, tfm_hi = bt.quantile(0.1)[:, H - 1], bt.quantile(0.9)[:, H - 1]

persistence_pred = cases[targets - H]   # Ch11 的 naive 基準（7 天後 = 最後已知值；等於週期 7 的 seasonal naive）


def mae(pred, idx=targets):
    return np.mean(np.abs(pred - cases[idx]))


def rmse(pred, idx=targets):
    return np.sqrt(np.mean((pred - cases[idx]) ** 2))


def mape(pred, idx=targets):
    return np.mean(np.abs(pred - cases[idx]) / np.maximum(cases[idx], 1e-6)) * 100


# 11_dl_sequence.ipynb 在同一個測試期印出的成績（MAE, RMSE, MAPE%）
CH11_REFERENCE = {
    "LSTM（Ch11，自己訓練）": (2.149, 2.727, 21.45),
    "1D-CNN（Ch11，自己訓練）": (2.197, 2.639, 20.42),
}

rows = [
    ("Persistence（naive 基準）", mae(persistence_pred), rmse(persistence_pred), mape(persistence_pred)),
    ("TimesFM 3.0（零樣本）", mae(tfm_pred), rmse(tfm_pred), mape(tfm_pred)),
] + [(name, *values) for name, values in CH11_REFERENCE.items()]
table = pd.DataFrame(rows, columns=["模型", "MAE", "RMSE", "MAPE(%)"]).round(3)
print(table.to_string(index=False))

print(f"\nTimesFM 80% 預測區間覆蓋率：{tsfm.coverage(cases[targets], tfm_lo, tfm_hi):.0%}（理想約 80%）")
print(f"TimesFM WQL（越小越好）：{tsfm.mean_weighted_quantile_loss(cases[targets], bt.quantiles[:, H - 1]):.3f}")
print(f"TimesFM 贏過 persistence？ {mae(tfm_pred) < mae(persistence_pred)}")
print(f"TimesFM 贏過 Ch11 的 LSTM？ {mae(tfm_pred) < CH11_REFERENCE['LSTM（Ch11，自己訓練）'][0]}")

In [ ]:
# --- Step 4b：測試期的 7 天後預測 ---
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(targets, cases[targets], color="#1A1A1A", linewidth=2, label="真實病例數")
ax.plot(targets, persistence_pred, color="#6B6B6B", linestyle="--", label="Persistence（naive）")
ax.plot(targets, tfm_pred, color="#D97757", linewidth=2, label="TimesFM 3.0（零樣本）")
ax.fill_between(targets, tfm_lo, tfm_hi, color="#D97757", alpha=0.2, label="TimesFM 80% 預測區間")
ax.set_xlabel("天數（day index）")
ax.set_ylabel("每日病例數")
ax.set_title("滾動起點回測：每一天都是「7 天前」做出的預測")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 怎麼讀這張成績單？

- **贏 persistence**：一行訓練程式都沒寫，TimesFM 的 MAE 就比 naive 基準低，而且還附帶覆蓋率接近 80% 的預測區間——這是零樣本模型最有價值的地方：**一個不用調教、立刻可用的強基準**
- **輸給 Ch11 的 LSTM / CNN**：它們是**在這條序列上訓練過**的模型，學到了「7 天前的氣溫推高病例數」這個機制；TimesFM 只看病例數本身的歷史形狀，不知道這個機制
- 這正是 Ch11 決策框架的精神：有足夠的歷史、又有明確的領先指標時，量身訓練的模型可以贏；**歷史很短、沒時間訓練、或要同時預測上百條序列時**，零樣本模型才是主場

### 可以調整 context 長度嗎？——用驗證期決定，不准偷看測試期

TimesFM 預設吃下全部歷史。只看最近幾週會不會更準？這是一個**超參數**，跟 Ch11 的 epoch 數一樣，**只能用驗證期決定**（Ch11 的驗證期：目標日 day 260–299），選好之後才碰測試期一次。

In [ ]:
# --- Step 5：在驗證期選 context 長度，再到測試期驗收一次 ---
val_origins = np.arange(split - 40 - H + 1, split - H + 1)   # 目標日 day 260–299（全部在測試期之前）
val_targets = val_origins + H - 1
candidates = [28, 56, 112, None]                              # None = 全部歷史

val_scores = {}
for max_context in candidates:
    ctx = tsfm.rolling_origin_contexts(cases, val_origins, max_context=max_context)
    res = tsfm.forecast_batch(model, ctx, H, cache_key=f"val_context_{max_context or 'all'}",
                              cache_file=CACHE, verbose=False)
    val_scores[max_context] = mae(res.forecast[:, H - 1], val_targets)
    print(f"context = {str(max_context or '全部'):>4} 天 → 驗證期 MAE = {val_scores[max_context]:.3f}")

best = min(val_scores, key=val_scores.get)
test_ctx = tsfm.rolling_origin_contexts(cases, origins, max_context=best)
tuned = tsfm.forecast_batch(model, test_ctx, H, cache_key=f"test_context_{best or 'all'}", cache_file=CACHE)
print(f"\n驗證期選出 context = {best or '全部'} → 測試期 MAE = {mae(tuned.forecast[:, H - 1]):.3f}"
      f"（預設全部歷史：{mae(tfm_pred):.3f}）")

## Step 6 — 共變量：把氣溫也告訴它

TimesFM 3.0 的新功能是**原生支援多變量與共變量**，不用針對任務調整：

- `past_only_covariates`：只知道到起點為止的變數（例如過去的氣溫），形狀 `(共變量數, T)`
- `past_future_covariates`：連未來都已知的變數（例如氣象預報、假日、已排定的疫苗接種日），形狀 `(共變量數, T + horizon)`
- 也可以一次丟多條**目標**序列（形狀 `(目標數, T)`，例如各區的病例數），模型會同時預測、彼此參考

這組資料的病例數受「7 天前氣溫」驅動，而預測 7 天後時，那天的「7 天前」正好是起點——所以只用**過去的**氣溫理論上就夠了。下面兩種都試：只給過去氣溫，以及假設有一份**完美的 7 天氣溫預報**。

In [ ]:
# --- Step 6：加入氣溫共變量（同樣 60 個起點）---
past_only = [temp[None, :o] for o in origins]          # 每個起點：過去的氣溫，形狀 (1, T)
past_future = [temp[None, :o + H] for o in origins]    # 每個起點：過去 + 未來 7 天的氣溫，形狀 (1, T + 7)

# tsfm.forecast_batch 在有模型時就是呼叫：
#   model.predict_batch(contexts, horizon=7, past_only_covariates=past_only, return_quantiles=True)
cov_po = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_temp_past_only",
                             cache_file=CACHE, past_only_covariates=past_only)
cov_pf = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_temp_past_future",
                             cache_file=CACHE, past_future_covariates=past_future)

rows = [
    ("TimesFM：只有病例數", bt),
    ("TimesFM：+ 過去氣溫", cov_po),
    ("TimesFM：+ 完美氣溫預報", cov_pf),
]
for name, res in rows:
    pred = res.forecast[:, H - 1]
    cov80 = tsfm.coverage(cases[targets], res.quantile(0.1)[:, H - 1], res.quantile(0.9)[:, H - 1])
    print(f"{name:<18} MAE = {mae(pred):.3f}   80% 區間覆蓋率 = {cov80:.0%}   （模型耗時 {res.seconds:.1f} 秒）")
print(f"{'LSTM（Ch11，有氣溫、自己訓練）':<18} MAE = {CH11_REFERENCE['LSTM（Ch11，自己訓練）'][0]:.3f}")

### 共變量沒有幫上忙——為什麼？

在這組資料上，加了氣溫的 TimesFM **沒有比只看病例數更準**（甚至略差）。誠實的解讀：

1. 零樣本模型不知道「氣溫高於 24 度才有作用、延遲剛好 7 天」這個**這組資料專屬**的機制；它只能從同一段歷史裡猜兩條序列的關係，而這裡氣溫的訊號被大量雜訊和每週節律蓋過
2. Ch11 的 LSTM 是**在這條序列上訓練**出來的，才學得到這個機制
3. 實務教訓：共變量不是加了就會變好——**每加一個共變量，都要用同樣的滾動起點回測證明它有用**；也別忘了「未來共變量」在真實世界裡本身就是預測（氣象預報也會錯）

## Step 7 — 換一條序列：Ch07 的 90 天類流感監測資料

Ch07 的 Part B 合成了一條 90 天的類流感每日通報數（趨勢 + 7 天週期 + 雜訊），用 ARIMA / SARIMA / Prophet 預測最後 7 天。同一條序列（同一個 seed），換 TimesFM 來預測：

- **(a) Ch07 原本的單一切分**：前 83 天 → 最後 7 天，直接跟 Ch07 印出的 MAE 比
- **(b) 滾動起點**：只看一週的成績太容易靠運氣，所以再做 14 個起點（day 56, 58, …, 82），每個預測未來 7 天，跟 **seasonal naive**（「下週 = 上週同一天」）和每個起點重新配適的 **SARIMA** 比

In [ ]:
# --- Step 7：Ch07 的 90 天類流感序列（跟 Ch07 的 synth_cases 完全相同）---
import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX

rng = np.random.default_rng(42)
n_days = 90
trend = np.linspace(3, 7, n_days)
seasonal = 3 * np.sin(2 * np.pi * np.arange(n_days) / 7)
noise = rng.normal(0, 1.2, n_days)
ili = np.maximum(0, (trend + seasonal + noise).round()).astype(int)

# (a) Ch07 的單一切分：前 83 天 → 最後 7 天
one = tsfm.forecast_batch(model, [ili[:-7]], 7, cache_key="ili_last_week", cache_file=CACHE)
CH07_REFERENCE = {"ARIMA(1,1,1)": 1.872, "SARIMA(1,1,1)(1,1,1,7)": 0.770, "Prophet": 0.774}  # Ch07 印出的 MAE
print("(a) 最後 7 天的 MAE（同一個切分）")
print(f"    TimesFM 3.0（零樣本）     {np.mean(np.abs(one.forecast[0] - ili[-7:])):.3f}")
print(f"    Seasonal naive（上週同日）{np.mean(np.abs(ili[-14:-7] - ili[-7:])):.3f}")
for name, value in CH07_REFERENCE.items():
    print(f"    {name:<24}{value:.3f}（Ch07）")

# (b) 滾動起點：14 個起點，每個預測未來 7 天
ili_origins = np.arange(56, 84, 2)
ili_actual = np.stack([ili[o:o + 7] for o in ili_origins])
roll = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(ili, ili_origins), 7,
                           cache_key="ili_rolling", cache_file=CACHE)
snaive = np.stack([ili[o - 7:o] for o in ili_origins])
sarima = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # 短序列上 SARIMA 常有收斂警告
    for o in ili_origins:
        fit = SARIMAX(ili[:o].astype(float), order=(1, 1, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
        sarima.append(fit.forecast(7))
sarima = np.array(sarima)

print(f"\n(b) 滾動起點（{len(ili_origins)} 個起點 × 7 天）的 MAE")
print(f"    TimesFM 3.0（零樣本）     {np.mean(np.abs(roll.forecast - ili_actual)):.3f}")
print(f"    SARIMA（每個起點重新配適）{np.mean(np.abs(sarima - ili_actual)):.3f}")
print(f"    Seasonal naive            {np.mean(np.abs(snaive - ili_actual)):.3f}")
print(f"    TimesFM 80% 區間覆蓋率：{tsfm.coverage(ili_actual, roll.quantile(0.1), roll.quantile(0.9)):.0%}")

ILI_READ_PLACEHOLDER

## Step 8 — 什麼時候**不能**相信它？

TSFM 學的是「時間序列通常長什麼樣子」——平滑、有週期、延續趨勢。疫情偏偏常常不照劇本走。下面用本書的主線資料示範兩個最危險的情境：

1. **群突發剛開始（outbreak onset）**：松柏護理之家的每日發病數，在第 5 天（高峰前夕）與第 9 天（剛過高峰）各預測一次未來 5 天
2. **通報延遲（reporting delay）**：最近幾天的病例還沒通報完整（Ch04 的通報延遲），模型會把「還沒補登」誤認成「疫情下降」

In [ ]:
# --- Step 8a：Legionella 群突發——零樣本模型看不到「即將爆發」---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
onset = (
    df[df["clinical_severity"] != "not_ill"]
    .groupby("symptom_onset_date").size()
    .asfreq("D", fill_value=0)
)
y = onset.to_numpy(dtype=float)
print(f"每日發病數（{len(y)} 天）：{y.astype(int).tolist()}")

leg_origins = [5, 9]   # 第 5 天：高峰前夕；第 9 天：剛過高峰
leg = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, leg_origins), 5,
                          cache_key="legionella_onset", cache_file=CACHE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, k, o in zip(axes, range(len(leg_origins)), leg_origins):
    future = np.arange(o, o + 5)
    ax.bar(np.arange(len(y)), y, color="#6A9BCC", alpha=0.5, label="實際每日發病數")
    ax.plot(future, leg.forecast[k], color="#D97757", marker="o", linewidth=2, label="TimesFM 中位數")
    ax.fill_between(future, leg.quantile(0.1)[k], leg.quantile(0.9)[k], color="#D97757", alpha=0.2,
                    label="80% 預測區間")
    ax.axvline(o - 0.5, color="#1A1A1A", linestyle="--")
    ax.set_title(f"起點：第 {o} 天（只看得到前 {o} 天）")
    ax.set_xlabel("發病第幾天")
    inside = tsfm.coverage(y[future], leg.quantile(0.1)[k], leg.quantile(0.9)[k])
    print(f"起點第 {o} 天：預測 {np.round(leg.forecast[k], 1)}，實際 {y[future].astype(int)}，"
          f"落在 80% 區間內 {inside:.0%}")
axes[0].set_ylabel("每日發病數")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 8b：通報延遲——最近 7 天還沒通報完整 ---
origin = 330
completeness = np.array([0.95, 0.9, 0.8, 0.65, 0.5, 0.35, 0.2])   # 最近 7 天已通報的比例（越近越不完整）
reported = cases[:origin].copy()
reported[-7:] = reported[-7:] * completeness                      # 「今天」看到的資料

delay = tsfm.forecast_batch(model, [cases[:origin], reported], 7, cache_key="reporting_delay", cache_file=CACHE)
truth = cases[origin:origin + 7]
print(f"未來 7 天真實病例數合計：            {truth.sum():6.1f}")
print(f"用完整資料預測的合計：                {delay.forecast[0].sum():6.1f}")
print(f"用「還沒補登完」的資料預測的合計：    {delay.forecast[1].sum():6.1f}"
      f"（低估 {1 - delay.forecast[1].sum() / truth.sum():.0%}）")
print("→ 解法不在模型：先做 nowcasting（校正通報延遲），或只用已經穩定的日期當歷史。")

SUMMARY_PLACEHOLDER